In [1]:
# =============================================================
# marketing_campaign.csv 전처리 (Colab)
# 팀 결정사항(데이터_전처리.md) 반영
# =============================================================
# 실행 전: Colab 좌측 파일 아이콘 -> marketing_campaign.csv 업로드
# 또는 아래 파일 업로드 위젯 사용

from google.colab import files
uploaded = files.upload()  # marketing_campaign.csv 선택

# -------------------------------------------------------------
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

df = pd.read_csv("marketing_campaign.csv", sep=";", encoding="utf-8-sig")
print(f"[0] 원본 적재: {df.shape[0]}행 x {df.shape[1]}열")
n_before = len(df)

# 단위경제성 상수 (컬럼 삭제 전에 값 보관 — 손익분기 계산에 계속 필요)
COST_PER_CONTACT = int(df["Z_CostContact"].iloc[0])   # 3
REVENUE_PER_ACCEPT = int(df["Z_Revenue"].iloc[0])     # 11
BREAKEVEN_RATE = COST_PER_CONTACT / REVENUE_PER_ACCEPT
print(f"    손익분기 반응률 = {COST_PER_CONTACT}/{REVENUE_PER_ACCEPT} = {BREAKEVEN_RATE:.1%}")

# -------------------------------------------------------------
# 1. 완전중복 처리 (2단계)
#    ID를 제외한 28개 컬럼이 일치하는 레코드가 다수 존재.
#    Recency/Income/Dt_Customer가 동시에 일치할 확률은 사실상 0
#    -> 우연이 아니라 적재 과정의 중복 삽입.
# -------------------------------------------------------------

# 1a. Response까지 완전히 같은 행 -> 한 개만 남김 (관련 358행 / 제거 182건)
dup_key_all = [c for c in df.columns if c != "ID"]
n_full_dup_rows = int(df.duplicated(subset=dup_key_all, keep=False).sum())
df = df.drop_duplicates(subset=dup_key_all, keep="first").reset_index(drop=True)
print(f"[1a] 완전중복(Response 포함) 제거: 관련 {n_full_dup_rows}행 중 "
      f"{n_before - len(df)}건 삭제 -> {len(df)}행")

# 1b. Response만 다른 경우 -> Response=1 행을 대표값으로 남기고 나머지 삭제
#     (팀 결정: 삭제 대신 "캠페인에 반응한 적이 있다"는 신호를 보존)
n_before_1b = len(df)
conf_key = [c for c in df.columns if c not in ("ID", "Response")]
df["_conf_grp_size"] = df.groupby(conf_key, dropna=False)["Response"].transform("nunique")
conflict_mask = df["_conf_grp_size"] > 1

# 상충 그룹 내에서 Response=1 우선, 동일 그룹 내 첫 행만 유지
df["_sort_key"] = -df["Response"]  # Response=1이 위로 오도록 정렬
df = df.sort_values("_sort_key").drop_duplicates(subset=conf_key, keep="first")
df = df.drop(columns=["_conf_grp_size", "_sort_key"]).reset_index(drop=True)

print(f"[1b] Response만 상충하는 중복: Response=1 행을 대표값으로 채택 "
      f"-> {n_before_1b}행 -> {len(df)}행 (삭제 {n_before_1b - len(df)}건)")
print(f"[1]  중복 처리 합계: {n_before}행 -> {len(df)}행 "
      f"(삭제 {n_before - len(df)}건, {(n_before - len(df)) / n_before:.1%})")

# -------------------------------------------------------------
# 2. 불필요한 상수 컬럼 제거
# -------------------------------------------------------------
df = df.drop(columns=["Z_CostContact", "Z_Revenue"])
print("[2] Z_CostContact, Z_Revenue 컬럼 삭제 (값은 위 BREAKEVEN_RATE로 보존)")

# -------------------------------------------------------------
# 3. Income 결측/이상치 처리
#    - 극단값(666666) 포함 전체를 중앙값으로 대체 (팀 결정: 전체 기준 중앙값)
#    - Income=7500이 12건 반복되나, 생년/학력/혼인상태/가입일이 전부 달라
#      중복 삽입이 아닌 하한값(설문 기본값 등)으로 판단 -> 그대로 유지, 주석만 남김
# -------------------------------------------------------------
df["Income_missing"] = df["Income"].isna().astype(int)
n_income_missing = int(df["Income_missing"].sum())

income_median = df["Income"].median()
df["Income"] = df["Income"].fillna(income_median)

n_income_outlier = int((df["Income"] > 200000).sum())
df.loc[df["Income"] > 200000, "Income"] = income_median

print(f"[3] Income: 결측 {n_income_missing}건 + 이상치(>200000) {n_income_outlier}건 "
      f"-> 전체 중앙값({income_median:,.0f})으로 대체")
print("    * Income=7500이 12건 존재하나 다른 모든 컬럼값이 서로 다름 "
      "-> 중복 삽입이 아닌 소득 하한값(설문 기본응답 등)으로 추정, 그대로 유지")

# -------------------------------------------------------------
# 4. Year_Birth -> Age 파생 (2014년 기준), 극단적 이상치 필터링
#    1893/1899/1900년생 3건은 2014년 기준 114~121세로 비현실적 -> 행 제거
# -------------------------------------------------------------
n_before_age = len(df)
df = df[df["Year_Birth"] >= 1940].reset_index(drop=True)
print(f"[4] Year_Birth<1940 (3건, 114~121세) 제거: {n_before_age} -> {len(df)}행")

df["Age"] = 2014 - df["Year_Birth"]

# 나중에 범주형으로 묶기 위한 연령대 구간 (필요 시 EDA/모델링 단계에서 사용)
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 30, 40, 50, 60, 70, 200],
    labels=["20대이하", "30대", "40대", "50대", "60대", "70대이상"],
)
print("    Age 파생 완료(2014년 기준), AgeGroup 범주형 컬럼 생성")

# -------------------------------------------------------------
# 5. Education 4개 범주로 정리 (2n Cycle -> Master 편입)
# -------------------------------------------------------------
df["Education"] = df["Education"].replace({"2n Cycle": "Master"})
print("[5] Education: 2n Cycle -> Master 편입, 잔존 범주:",
      sorted(df["Education"].unique()))

# -------------------------------------------------------------
# 6. Marital_Status 정리
#    Alone, Absurd, YOLO -> Single 로 병합 (표본 3+2+2=7건, 정상 범주와 병합)
#    Married, Together, Single, Divorced, Widow 는 그대로 유지
# -------------------------------------------------------------
marital_map = {"Alone": "Single", "Absurd": "Single", "YOLO": "Single"}
n_recode = int(df["Marital_Status"].isin(marital_map).sum())
df["Marital_Status"] = df["Marital_Status"].replace(marital_map)
print(f"[6] Marital_Status 재코딩 {n_recode}건 -> Single 병합, "
      f"잔존 범주: {sorted(df['Marital_Status'].unique())}")

# -------------------------------------------------------------
# 7. Dt_Customer 날짜형 변환
# -------------------------------------------------------------
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="mixed")
print("[7] Dt_Customer datetime 변환 완료, 범위:",
      df["Dt_Customer"].min().date(), "~", df["Dt_Customer"].max().date())

# -------------------------------------------------------------
# 8. 파생변수
#    - Children = Kidhome + Teenhome
#    - AnyAccepted = 과거 캠페인(1~5) 중 하나라도 수락 여부
# -------------------------------------------------------------
df["Children"] = df["Kidhome"] + df["Teenhome"]

cmp_cols = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
            "AcceptedCmp4", "AcceptedCmp5"]
df["AnyAccepted"] = (df[cmp_cols].sum(axis=1) > 0).astype(int)

print("[8] 파생변수 생성: Children(Kidhome+Teenhome), "
      "AnyAccepted(과거 캠페인 1개 이상 수락 여부)")

# -------------------------------------------------------------
# 9. 저장 및 요약
# -------------------------------------------------------------
df.to_csv("marketing_clean.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 60)
print(f"최종: {df.shape[0]}행 x {df.shape[1]}열 "
      f"(원본 {n_before}행 대비 {(n_before - df.shape[0]) / n_before:.1%} 손실)")
print(f"결측 잔존: {int(df.isna().sum().sum())}건")
print(f"전체 반응률: {df['Response'].mean():.2%}  |  손익분기: {BREAKEVEN_RATE:.1%}")
ct = pd.crosstab(df["AnyAccepted"], df["Response"], normalize="index")
print(f"AnyAccepted=1 반응률: {ct.loc[1,1]:.2%}  /  =0: {ct.loc[0,1]:.2%}  "
      f"(격차 {ct.loc[1,1]-ct.loc[0,1]:.2%}p)")
print("=" * 60)

df.head()

# -------------------------------------------------------------
# 10. Colab에서 결과 파일 다운로드
# -------------------------------------------------------------
files.download("marketing_clean.csv")

Saving marketing_campaign.csv to marketing_campaign.csv
[0] 원본 적재: 2240행 x 29열
    손익분기 반응률 = 3/11 = 27.3%
[1a] 완전중복(Response 포함) 제거: 관련 358행 중 182건 삭제 -> 2058행
[1b] Response만 상충하는 중복: Response=1 행을 대표값으로 채택 -> 2058행 -> 2039행 (삭제 19건)
[1]  중복 처리 합계: 2240행 -> 2039행 (삭제 201건, 9.0%)
[2] Z_CostContact, Z_Revenue 컬럼 삭제 (값은 위 BREAKEVEN_RATE로 보존)
[3] Income: 결측 24건 + 이상치(>200000) 1건 -> 전체 중앙값(51,537)으로 대체
    * Income=7500이 12건 존재하나 다른 모든 컬럼값이 서로 다름 -> 중복 삽입이 아닌 소득 하한값(설문 기본응답 등)으로 추정, 그대로 유지
[4] Year_Birth<1940 (3건, 114~121세) 제거: 2039 -> 2036행
    Age 파생 완료(2014년 기준), AgeGroup 범주형 컬럼 생성
[5] Education: 2n Cycle -> Master 편입, 잔존 범주: ['Basic', 'Graduation', 'Master', 'PhD']
[6] Marital_Status 재코딩 6건 -> Single 병합, 잔존 범주: ['Divorced', 'Married', 'Single', 'Together', 'Widow']
[7] Dt_Customer datetime 변환 완료, 범위: 2012-07-30 ~ 2014-06-29
[8] 파생변수 생성: Children(Kidhome+Teenhome), AnyAccepted(과거 캠페인 1개 이상 수락 여부)

최종: 2036행 x 32열 (원본 2240행 대비 9.1% 손실)
결측 잔존: 0건
전체 반응률: 15.37%  |  손익분기: 27.3%
AnyAccepted

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>